In [ ]:
import polars as pl

# POLARS API

In [ ]:
df = pl.read_parquet("data/2019-Oct-10M.parquet")
df.collect_schema()

In [ ]:
df.head()

In [ ]:
df.describe()

## Selection data

In [ ]:
df.select("event_time").head()  # Single column
df.select(["event_type", "price"]).head()  # Multiple columns
# df['event_type'] # Select a Series, like Pandas Column

In [ ]:
# Selecting with transformation
df.select(
    pl.col("price").median().alias("median_price"),
    pl.col("brand").n_unique().alias("n_unique_brands"),
    pl.col("brand").is_not_null().sum(),
    pl.col("brand").str.len_chars().max().alias("max_brand_length"),
)

In [ ]:
df.get_column("brand").value_counts(sort=True).head()
# df['brand'].value_counts(sort=True) # Same way to access column, not recommended

## Filtering data

In [ ]:
df.filter(pl.col("price") > 200).head()
df.filter((pl.col("event_type") == "view") & (pl.col("brand") == "lenovo")).head()

## Transformation and grouping

In [ ]:
df.with_columns(pl.col("price").median().over("brand").alias("median_price_over_brand")).head()

In [ ]:
df.with_columns(
    pl.col("price").median().over("brand").alias("median_price_over_brand"),
    pl.col("event_time").str.to_datetime(format="%Y-%m-%d %H:%M:%S %Z"),
).group_by("event_type", "product_id", "category_id", "category_code", "brand", "price", "median_price_over_brand").agg(
    pl.col("user_id"),  # Grouping without real agg, derives in a list
    pl.col("event_time").dt.year().min().alias("year"),
).head()

### Eager vs Lazy Performance

In [ ]:
## NOTE: Restart kernel and clean cells at this point to free memory
import polars as pl

In [ ]:
df = pl.read_parquet("data/2019-Oct-10M.parquet")  # Eager read

In [ ]:
df.with_columns(pl.col("price").median().over("brand").alias("median_price_over_brand")).write_parquet(
    "data/2019-Oct-10M_eager.parquet"
)

In [ ]:
del df

In [ ]:
df = pl.scan_parquet("data/2019-Oct-10M.parquet")

In [ ]:
df.with_columns(pl.col("price").median().over("brand").alias("median_price_over_brand")).sink_parquet(
    "data/2019-Oct-10M_eager.parquet", engine="streaming"
)

In [ ]:
df_lazy_brand = pl.scan_parquet("data/2019-Oct-10M.parquet").select("brand").unique()

In [ ]:
df_lazy = (
    pl.scan_parquet("data/2019-Oct.parquet")
    .join(df_lazy_brand, on="brand", how="inner")
    .with_columns(pl.col("price").median().over("brand").alias("median_price_over_brand"))
    .group_by("event_type", "product_id", "category_id", "category_code", "brand", "price", "median_price_over_brand")
    .agg(pl.col("user_id"))
    .filter(pl.col("product_id") > 100000)
)

In [ ]:
df_lazy.show_graph(optimized=False)

In [ ]:
df_lazy.show_graph(optimized=True)

In [ ]:
df_lazy.collect(engine="streaming", no_optimization=True).write_parquet("data/2019-Oct-10M_lazy_no_opt.parquet")

In [ ]:
df_lazy.collect(engine="streaming").write_parquet("data/2019-Oct-10M_lazy_opt.parquet")

## Arrow interoperability

In [ ]:
df = pl.read_parquet("data/2019-Oct-10M.parquet")

In [ ]:
df.estimated_size("mb")

In [ ]:
df_pd = df.to_pandas(use_pyarrow_extension_array=False)
df_pd.head()

In [ ]:
df_pd_arrow = df.to_pandas(use_pyarrow_extension_array=True)
df_pd_arrow.head()

In [ ]:
import duckdb

conn = duckdb.connect("test.duckdb")

In [ ]:
conn.sql("CREATE OR REPLACE TABLE pandas_backend as SELECT * from df_pd")

In [ ]:
conn.sql("CREATE OR REPLACE  TABLE pandas_arrow as SELECT * from df_pd_arrow")

In [ ]:
conn.sql("CREATE OR REPLACE TABLE polars as SELECT * from df")

In [ ]:
conn.close()

## Open Source table formats integration

In [ ]:
# Iceberg integration
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, IntegerType, FloatType, LongType
from pyiceberg.catalog import load_catalog
import os


current_dir = os.path.abspath(".")
warehouse_path = f"file://{current_dir}/warehouse"
uri = f"sqlite:///{current_dir}/pyiceberg_catalog.db"

catalog = load_catalog("default", **{"type": "sql", "uri": uri, "warehouse": warehouse_path})

schema = Schema(
    NestedField(1, "event_time", StringType(), required=False),
    NestedField(2, "event_type", StringType(), required=False),
    NestedField(3, "product_id", IntegerType(), required=False),
    NestedField(4, "category_id", LongType(), required=False),
    NestedField(5, "category_code", StringType(), required=False),
    NestedField(6, "brand", StringType(), required=False),
    NestedField(7, "price", FloatType(), required=False),
    NestedField(8, "user_id", IntegerType(), required=False),
    NestedField(9, "user_session", StringType(), required=False),
)

In [ ]:
catalog.create_namespace("polars")
table = catalog.create_table("polars.2019_oct_10M", schema=schema)
df.write_iceberg(table, mode="append")

In [ ]:
pl.scan_iceberg(table).head().collect()

In [ ]:
# Delta Lake integration
df.write_delta("delta_table", mode="overwrite")

In [ ]:
pl.scan_delta("delta_table").head().collect()